# Retriever pondéré (Jaccard + attention AST + IDF) — comparaison à 4 signaux x 2 modèles

Notebook **séparé** de `colab_50_trials.ipynb` — ne le modifie pas, ne dépend pas de son état. Autonome comme lui : télécharge ses propres données depuis `microsoft/CodeT`, écrit son propre code sur disque.

**4 signaux de retrieval comparés** (mêmes fenêtres AST, seul le score change) :
1. Sans retrieval
2. Jaccard sur texte brut (RG1 officiel)
3. Jaccard sur identifiants AST (hérités du scope-mapping, ce qu'on avait déjà)
4. **Jaccard pondéré + attention AST** (nouveau : IDF précalculé x attention `exp(-lambda*distance_ast)` calculée par vrais sauts dans l'arbre via LCA, poids différenciés variable/import)

**2 modèles** : `codegen-2B-mono` (base, complétion brute) et `Qwen2.5-Coder-3B-Instruct` (instruct, via chat template) — les deux déjà validés dans `colab_50_trials.ipynb` avec leurs corrections respectives (`trim_code` pour la troncature, `strip_markdown_fence` pour les balises markdown de l'instruct).

**Important** : si une cellule plante avec une erreur CUDA ("device-side assert"), le GPU reste corrompu pour le reste de la session — **Exécution > Redémarrer la session** avant de relancer.

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Installer les dépendances

In [ ]:
!pip install -q scikit-learn editdistance transformers accelerate tree-sitter tree-sitter-python


## 1bis. (Optionnel) Token Hugging Face

Pas nécessaire pour ces modèles publics — évite juste le throttling anonyme si tu en as un.

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé.")
except Exception:
    print("Pas de secret HF_TOKEN configuré — pas grave, pas nécessaire pour ces modèles publics.")

## 2. Réécrire le code du projet (copié tel quel depuis `repocoder-mine/`)

In [ ]:
%%writefile dataset.py
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Literal


REQUIRED_FIELDS = {"prompt", "groundtruth", "right_context"}
Split = Literal[
    "baseline",
    "bm25",
    "unixcoder",
    "openai",
    "oracle_bm25",
    "oracle_unixcoder",
    "oracle_openai",
]


def load_jsonl(file_path: str | Path) -> list[dict[str, Any]]:
    """Charger les exemples valides d'un fichier JSONL."""
    records = []
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                print(f"Ligne ignorée {line_number}: JSON invalide ({error})")
                continue

            if not isinstance(record, dict):
                print(f"Ligne ignorée {line_number}: l'objet n'est pas un dictionnaire")
                continue

            missing_fields = REQUIRED_FIELDS - record.keys()
            if missing_fields:
                print(
                    f"Ligne ignorée {line_number}: champs manquants "
                    f"{sorted(missing_fields)}"
                )
                continue

            if any(not isinstance(record[field], str) for field in REQUIRED_FIELDS):
                print(f"Ligne ignorée {line_number}: un champ contient une valeur invalide")
                continue

            records.append(record)

    return records


def load_cceval_examples(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
):
    """Charger les exemples avec le modèle officiel de CrossCodeEval."""
    from cceval.dataset import load_cceval_dataset as load_official_dataset

    return load_official_dataset(
        path=str(path) if path is not None else None,
        language=language,
        split=split,
        sample=sample,
        seed=seed,
    )


def load_cceval_dataset(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
) -> list[dict[str, Any]]:
    """Charger le dataset CrossCodeEval avec ses champs d'évaluation."""
    if path is None:
        path = resolve_cceval_path(split, language)

    records = load_jsonl(path)
    task_ids = [record.get("metadata", {}).get("task_id") for record in records]
    task_ids = [task_id for task_id in task_ids if task_id is not None]
    if len(task_ids) != len(set(task_ids)):
        raise ValueError("Le dataset contient des task_id en double")

    if sample is not None:
        if sample < 0:
            raise ValueError("sample doit être positif")
        records = random.Random(seed).sample(records, min(sample, len(records)))

    return records


def resolve_cceval_path(split: Split = "baseline", language: str = "python") -> Path:
    """Construire le chemin CrossCodeEval depuis CCEVAL_DATA_DIR."""
    data_dir = os.environ.get("CCEVAL_DATA_DIR")
    if data_dir is None:
        raise ValueError("La variable CCEVAL_DATA_DIR n'est pas définie")

    filenames = {
        "baseline": "line_completion.jsonl",
        "bm25": "line_completion_rg1_bm25.jsonl",
        "unixcoder": "line_completion_rg1_unixcoder_cosine_sim.jsonl",
        "openai": "line_completion_rg1_openai_cosine_sim.jsonl",
        "oracle_bm25": "line_completion_oracle_bm25.jsonl",
        "oracle_unixcoder": "line_completion_oracle_unixcoder_cosine_sim.jsonl",
        "oracle_openai": "line_completion_oracle_openai_cosine_sim.jsonl",
    }
    return Path(data_dir) / language / filenames[split]


SLIDING_WINDOW_SIZE = 20  # S_w dans l'article RepoCoder
SLIDING_STRIDE = 10  # S_s dans l'article RepoCoder


def slide_over_text(
    text: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[str]:
    """Découper un texte en fenêtres glissantes de lignes (S_w, S_s de RepoCoder)."""
    if window_size <= 0:
        raise ValueError("window_size doit être supérieur à 0")
    if stride <= 0 or stride > window_size:
        raise ValueError("stride doit être compris entre 1 et window_size")

    lines = text.splitlines(keepends=True)
    if not lines:
        return []

    windows = []
    for start in range(0, len(lines), stride):
        window_text = "".join(lines[start:start + window_size]).strip()
        if window_text:
            windows.append(window_text)
        if start + window_size >= len(lines):
            break
    return windows


def last_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n dernières lignes d'un texte (utilisé comme requête de retrieval)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[-n:])


def first_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n premières lignes d'un texte (utilisé sur la prédiction précédente)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[:n])


def extract_repository_snippets(
    records: list[dict[str, Any]],
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, list[dict[str, Any]]]:
    """Regrouper les exemples CCEval par dépôt puis extraire leurs fenêtres glissantes.

    Reproduit le découpage de RepoCoder (S_w=20, S_s=10 par défaut) : chaque
    dépôt (metadata.repository) reçoit la liste des morceaux de code obtenus
    en faisant glisser une fenêtre sur les lignes de chaque exemple.
    """
    repositories: dict[str, list[dict[str, Any]]] = {}

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if not repository:
            continue

        file_content = record["prompt"] + record.get("right_context", "")
        snippets = slide_over_text(file_content, window_size=window_size, stride=stride)

        for snippet in snippets:
            repositories.setdefault(repository, []).append(
                {
                    "task_id": metadata.get("task_id"),
                    "file": metadata.get("file"),
                    "snippet": snippet,
                }
            )

    return repositories


def extract_cceval_repository_snippets(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    sample: int | None = None,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Charger le dataset CCEval puis extraire les fenêtres glissantes par dépôt.

    Utilise par défaut les hyperparamètres de l'article RepoCoder
    (S_w=20, S_s=10) pour construire la base de code de chaque dépôt.
    """
    records = load_cceval_dataset(
        path=path, language=language, split=split, sample=sample, seed=seed
    )
    return extract_repository_snippets(records, window_size=window_size, stride=stride)


def save_repository_snippets(
    repositories: dict[str, list[dict[str, Any]]], output_dir: str | Path
) -> dict[str, int]:
    """Sauvegarder les fenêtres glissantes de chaque dépôt dans son propre fichier JSONL."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    counts = {}
    for repository, snippets in repositories.items():
        safe_name = re.sub(r"[^\w.-]", "_", repository)
        save_jsonl(snippets, output_path / f"{safe_name}.jsonl")
        counts[repository] = len(snippets)

    return counts


def prepare_repository_snippets(
    output_dir: str | Path,
    path: str | Path | None = None,
    language: str = "python",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, int]:
    """Extraire les fenêtres glissantes du split baseline (line_completion.jsonl) et les sauvegarder par dépôt."""
    repositories = extract_cceval_repository_snippets(
        path=path,
        language=language,
        split="baseline",
        window_size=window_size,
        stride=stride,
    )
    return save_repository_snippets(repositories, output_dir)


def split_dataset(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Mélanger les exemples puis créer les ensembles train/validation/test."""
    if train_ratio <= 0 or validation_ratio < 0:
        raise ValueError("Les ratios doivent être positifs")
    if train_ratio + validation_ratio >= 1:
        raise ValueError("La somme des ratios doit être inférieure à 1")

    shuffled_records = records.copy()
    random.Random(seed).shuffle(shuffled_records)

    train_end = int(len(shuffled_records) * train_ratio)
    validation_end = train_end + int(len(shuffled_records) * validation_ratio)

    return {
        "train": shuffled_records[:train_end],
        "validation": shuffled_records[train_end:validation_end],
        "test": shuffled_records[validation_end:],
    }


def split_by_repository(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Séparer les dépôts entiers pour éviter une fuite entre les ensembles."""
    repositories = {}
    records_without_repository = []

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if repository:
            repositories.setdefault(repository, []).append(record)
        else:
            records_without_repository.append(record)

    if not repositories:
        return split_dataset(records, train_ratio, validation_ratio, seed)

    repository_names = list(repositories)
    random.Random(seed).shuffle(repository_names)
    train_repository_end = max(1, int(len(repository_names) * train_ratio))
    validation_repository_end = train_repository_end + int(
        len(repository_names) * validation_ratio
    )

    splits = {
        "train": [],
        "validation": [],
        "test": [],
    }
    for repository in repository_names[:train_repository_end]:
        splits["train"].extend(repositories[repository])
    for repository in repository_names[train_repository_end:validation_repository_end]:
        splits["validation"].extend(repositories[repository])
    for repository in repository_names[validation_repository_end:]:
        splits["test"].extend(repositories[repository])

    # Les exemples sans dépôt sont répartis uniquement après le découpage principal.
    splits["train"].extend(records_without_repository)
    return splits


def save_jsonl(records: list[dict[str, Any]], file_path: str | Path) -> None:
    """Sauvegarder une liste d'exemples au format JSONL."""
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def prepare_dataset(input_path: str | Path, output_dir: str | Path) -> dict[str, int]:
    """Charger, séparer et sauvegarder le dataset dans trois fichiers."""
    records = load_jsonl(input_path)
    splits = split_by_repository(records)
    output_path = Path(output_dir)

    for split_name, split_records in splits.items():
        save_jsonl(split_records, output_path / f"{split_name}.jsonl")

    return {split_name: len(split_records) for split_name, split_records in splits.items()}


if __name__ == "__main__":
    project_dir = Path(__file__).resolve().parent
    source = Path(r"C:\Users\User\Downloads\line_completion.jsonl")
    destination = project_dir / "data"
    counts = prepare_dataset(source, destination)

    print("Dataset organisé :")
    for split_name, count in counts.items():
        print(f"- {split_name}: {count} exemples")

    # Première extraction : fenêtres glissantes (S_w=20, S_s=10) par dépôt,
    # à partir de line_completion.jsonl uniquement, sauvegardées dans un dossier dédié.
    repository_destination = project_dir / "data" / "repositories"
    repository_counts = prepare_repository_snippets(repository_destination, path=source)

    print(f"\nFenêtres glissantes sauvegardées dans {repository_destination} :")
    for repository, count in repository_counts.items():
        print(f"- {repository}: {count} fenêtres")

In [ ]:
%%writefile ast_chunker.py
import ast
import hashlib
import os
import pickle
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dataset import SLIDING_STRIDE, SLIDING_WINDOW_SIZE

PROJECT_DIR = Path(__file__).resolve().parent
DEFAULT_CACHE_DIR = PROJECT_DIR / "data" / "cache" / "ast_chunks"


@dataclass
class ScopeBlock:
    """Un bloc nommé (classe ou fonction/méthode) avec ses bornes de lignes et ses identifiants propres."""

    kind: str  # "class" ou "function"
    name: str
    line_start: int
    line_end: int
    identifiers: set[str] = field(default_factory=set)


class _LocalNamesCollector(ast.NodeVisitor):
    """Collecte les noms assignés/définis dans un scope, sans descendre dans les classes/fonctions imbriquées
    (elles ont leur propre ScopeBlock)."""

    def __init__(self) -> None:
        self.names: set[str] = set()

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self.names.add(node.name)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self.names.add(node.name)

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        self.names.add(node.name)

    def visit_arg(self, node: ast.arg) -> None:
        self.names.add(node.arg)

    def visit_Name(self, node: ast.Name) -> None:
        if isinstance(node.ctx, ast.Store):
            self.names.add(node.id)

    def visit_ExceptHandler(self, node: ast.ExceptHandler) -> None:
        if node.name:
            self.names.add(node.name)
        self.generic_visit(node)

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name.split(".")[0])

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name)


def _collect_local_names(node: ast.AST) -> set[str]:
    """Noms locaux à un bloc (paramètres, variables assignées, imports locaux, fonctions/classes imbriquées)."""
    collector = _LocalNamesCollector()
    collector.generic_visit(node)  # generic_visit: visite les enfants, pas node lui-même
    return collector.names


def _collect_class_attributes(class_node: ast.ClassDef) -> set[str]:
    """Attributs de classe (`x = 1` dans le corps) et d'instance (`self.x = ...` dans les méthodes)."""
    attributes: set[str] = set()

    for stmt in class_node.body:
        if isinstance(stmt, ast.Assign):
            for target in stmt.targets:
                if isinstance(target, ast.Name):
                    attributes.add(target.id)
        elif isinstance(stmt, ast.AnnAssign) and isinstance(stmt.target, ast.Name):
            attributes.add(stmt.target.id)

    for node in ast.walk(class_node):
        if isinstance(node, ast.Attribute) and isinstance(node.ctx, ast.Store) and isinstance(node.value, ast.Name):
            attributes.add(node.attr)

    return attributes


def build_scope_map(code: str) -> tuple[set[str], list[ScopeBlock]]:
    """Analyser le code d'un fichier et retourner (imports du module, blocs classes/fonctions).

    Les imports de haut niveau (pas dans une fonction/classe) sont visibles dans
    tout le fichier. Chaque classe et chaque fonction/méthode devient un
    ScopeBlock avec ses propres identifiants (nom, attributs/paramètres,
    variables locales) et ses bornes de lignes (`node.lineno`/`node.end_lineno`).
    """
    tree = ast.parse(code)
    module_imports: set[str] = set()
    blocks: list[ScopeBlock] = []

    def visit(node: ast.AST, inside_def: bool) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.Import, ast.ImportFrom)) and not inside_def:
                if isinstance(child, ast.Import):
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name.split(".")[0])
                else:
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name)

            if isinstance(child, ast.ClassDef):
                own_methods = {
                    n.name for n in child.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                }
                identifiers = {child.name} | _collect_class_attributes(child) | own_methods
                blocks.append(ScopeBlock("class", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                identifiers = {child.name} | _collect_local_names(child)
                blocks.append(ScopeBlock("function", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            else:
                visit(child, inside_def)

    visit(tree, False)
    return module_imports, blocks


def chunk_file_ast(
    file_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Découper un fichier Python en fenêtres glissantes de lignes enrichies par l'AST.

    Chaque fenêtre hérite des identifiants de TOUS les blocs (classe et/ou
    fonction(s)) dont l'intervalle de lignes chevauche la fenêtre, plus les
    imports du module. Une fenêtre à cheval sur deux méthodes hérite ainsi
    des identifiants des deux, afin qu'une requête touchant l'une ou l'autre
    puisse retrouver ce morceau.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            code = file.read()
    except OSError as error:
        print(f"Erreur de lecture de {file_path}: {error}")
        return []

    try:
        module_imports, blocks = build_scope_map(code)
    except SyntaxError as error:
        print(f"Fichier ignoré (syntaxe invalide) {file_path}: {error}")
        return []

    lines = code.splitlines()
    if not lines:
        return []

    chunks = []
    for start in range(0, len(lines), stride):
        end = min(start + window_size, len(lines))
        raw_code = "\n".join(lines[start:end]).strip()

        if raw_code:
            line_start, line_end = start + 1, end  # lignes 1-indexées, comme node.lineno

            identifiers = set(module_imports)
            for block in blocks:
                if max(line_start, block.line_start) <= min(line_end, block.line_end):
                    identifiers |= block.identifiers

            chunks.append(
                {
                    "file_path": file_path,
                    "line_start": line_start,
                    "line_end": line_end,
                    "raw_code": raw_code,
                    "identifiers": sorted(identifiers),
                }
            )

        if end >= len(lines):
            break

    return chunks


def load_and_chunk_repo_ast(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Parcourir tous les fichiers .py d'un dossier et produire les chunks enrichis par AST."""
    all_chunks: list[dict[str, Any]] = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                all_chunks.extend(chunk_file_ast(file_path, window_size=window_size, stride=stride))
    return all_chunks


def _repo_fingerprint(dir_path: str) -> str:
    """Empreinte du contenu d'un dossier (chemin + date de modif + taille de chaque .py).

    Sert à invalider le cache automatiquement si un fichier source a changé,
    sans avoir à relire/hacher le contenu de chaque fichier.
    """
    entries = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                stat = os.stat(file_path)
                entries.append((os.path.relpath(file_path, dir_path), stat.st_mtime_ns, stat.st_size))
    entries.sort()
    return hashlib.sha1(repr(entries).encode("utf-8")).hexdigest()


def _cache_path(dir_path: str, window_size: int, stride: int, cache_dir: str | Path) -> Path:
    safe_name = re.sub(r"[^\w.-]", "_", os.path.normpath(os.path.abspath(dir_path)))
    return Path(cache_dir) / f"{safe_name}_ws{window_size}_stride{stride}.pkl"


def load_and_chunk_repo_ast_cached(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    cache_dir: str | Path = DEFAULT_CACHE_DIR,
) -> list[dict[str, Any]]:
    """Comme `load_and_chunk_repo_ast`, mais met le résultat en cache sur disque.

    Le cache est invalidé automatiquement si un fichier .py du dossier a été
    ajouté/modifié/supprimé depuis la dernière exécution (voir `_repo_fingerprint`),
    ou si `window_size`/`stride` changent (chaque combinaison a son propre fichier
    de cache).
    """
    cache_file = _cache_path(dir_path, window_size, stride, cache_dir)
    fingerprint = _repo_fingerprint(dir_path)

    if cache_file.exists():
        with cache_file.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("fingerprint") == fingerprint:
            return cached["chunks"]

    chunks = load_and_chunk_repo_ast(dir_path, window_size=window_size, stride=stride)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with cache_file.open("wb") as file:
        pickle.dump({"fingerprint": fingerprint, "chunks": chunks}, file)
    return chunks


if __name__ == "__main__":
    project_dir = os.path.dirname(os.path.abspath(__file__))
    demo_file = os.path.join(project_dir, "retriever.py")
    chunks = chunk_file_ast(demo_file)

    print(f"{len(chunks)} morceaux générés depuis {demo_file}\n")
    for chunk in chunks[:5]:
        print(f"--- lignes {chunk['line_start']}-{chunk['line_end']} ---")
        print("identifiants:", chunk["identifiers"])
        print()

In [ ]:
%%writefile ast_distance.py
"""Distance de sauts dans l'arbre AST, pour calculer l'attention statique
(attention_score = exp(-lambda * distance_ast)) attendue par
weighted_ast_scorer.weighted_ast_attention_score.

Module autonome : parse le VRAI fichier source complet (pas un extrait
tronqué, souvent syntaxiquement invalide isolément) et mesure la distance
entre l'occurrence la plus proche (avant le curseur) d'une variable et le
noeud englobant le curseur, via leur plus proche ancêtre commun (LCA).
"""

import ast
import math


def annotate_tree(root: ast.AST) -> tuple[dict[int, ast.AST], dict[int, int]]:
    """Parcourt l'arbre une fois et retourne (parent, depth), indexés par id(node).

    id(node) plutôt que node directement : les noeuds ast ne sont pas hashables
    de façon fiable pour tous les types, id() est stable et rapide.
    """
    parent: dict[int, ast.AST] = {}
    depth: dict[int, int] = {}
    stack: list[tuple[ast.AST, ast.AST | None, int]] = [(root, None, 0)]
    while stack:
        node, par, d = stack.pop()
        parent[id(node)] = par
        depth[id(node)] = d
        for child in ast.iter_child_nodes(node):
            stack.append((child, node, d + 1))
    return parent, depth


def find_cursor_node(root: ast.AST, depth: dict[int, int], line_no: int) -> ast.AST:
    """Le noeud le plus profond (le plus spécifique) dont l'intervalle de lignes
    contient line_no — représente \"où se trouve le curseur\" dans l'arbre."""
    best = root
    best_depth = -1
    for node in ast.walk(root):
        node_start = getattr(node, "lineno", None)
        node_end = getattr(node, "end_lineno", None)
        if node_start is not None and node_end is not None and node_start <= line_no <= node_end:
            d = depth[id(node)]
            if d > best_depth:
                best = node
                best_depth = d
    return best


def find_closest_occurrence(root: ast.AST, var_name: str, line_no: int) -> ast.AST | None:
    """La dernière occurrence (Name ou arg) de var_name strictement avant line_no.

    \"Dernière avant le curseur\" = la définition/usage le plus probablement
    pertinent pour compléter le code à cet endroit.
    """
    best_node = None
    best_line = -1
    for node in ast.walk(root):
        name = None
        if isinstance(node, ast.Name) and node.id == var_name:
            name = node.id
        elif isinstance(node, ast.arg) and node.arg == var_name:
            name = node.arg
        if name is None:
            continue
        node_line = getattr(node, "lineno", None)
        if node_line is not None and node_line <= line_no and node_line > best_line:
            best_line = node_line
            best_node = node
    return best_node


def tree_distance(
    node_a: ast.AST, node_b: ast.AST, parent: dict[int, ast.AST], depth: dict[int, int]
) -> int:
    """Nombre de sauts entre deux noeuds dans l'arbre, via leur plus proche
    ancêtre commun (LCA) : depth(a) + depth(b) - 2*depth(LCA)."""
    ancestors_a = set()
    node = node_a
    while node is not None:
        ancestors_a.add(id(node))
        node = parent.get(id(node))

    node = node_b
    steps_b = 0
    while node is not None and id(node) not in ancestors_a:
        node = parent.get(id(node))
        steps_b += 1

    if node is None:
        # pas d'ancêtre commun trouvé (ne devrait pas arriver dans un même arbre
        # connexe) -> distance maximale plutôt que planter
        return depth[id(node_a)] + depth[id(node_b)]

    lca_depth = depth[id(node)]
    return (depth[id(node_a)] - lca_depth) + steps_b


def compute_query_vars_with_attention(
    file_source: str,
    line_no: int,
    candidate_vars: set[str],
    lam: float = 0.1,
) -> dict[str, float]:
    """Pour chaque variable candidate présente avant line_no dans le vrai fichier,
    calcule attention_score = exp(-lambda * distance_ast) où distance_ast est la
    distance de sauts (LCA) entre sa dernière occurrence et le noeud du curseur.

    Une variable candidate absente du fichier (ou jamais utilisée avant line_no)
    est simplement omise du résultat (pas de score par défaut).
    """
    root = ast.parse(file_source)
    parent, depth = annotate_tree(root)
    cursor_node = find_cursor_node(root, depth, line_no)

    query_vars: dict[str, float] = {}
    for var_name in candidate_vars:
        occurrence = find_closest_occurrence(root, var_name, line_no)
        if occurrence is None:
            continue
        distance = tree_distance(occurrence, cursor_node, parent, depth)
        query_vars[var_name] = math.exp(-lam * distance)

    return query_vars

In [ ]:
%%writefile weighted_ast_scorer.py
"""Scoring par Théorie des Ensembles Pondérés avec Attention Statique AST.

Module autonome, non branché au pipeline existant (retriever.py) — pour
tester la formule avant intégration éventuelle. Vrai Jaccard pondéré
(intersection / union) : IDF précalculé x score d'attention AST (déjà
fourni par l'appelant, pas recalculé ici) x poids fixe selon le type de
symbole (variable de portée ou import) côté requête ; IDF seul côté
symboles présents uniquement dans le chunk.
"""


def weighted_ast_attention_score(
    query_vars: dict[str, float],
    query_imports: set[str],
    chunk_symbols: set[str],
    chunk_imports: set[str],
    doc_weights: dict[str, float],
    var_weight: float = 2.0,
    import_weight: float = 2.5,
) -> float:
    """Jaccard pondéré entre une requête et un chunk candidat.

    query_vars : {nom_variable: attention_score}, attention_score déjà
        calculé en amont comme exp(-lambda * distance_ast) (proximité dans
        l'arbre AST par rapport au curseur) — cette fonction ne fait que le
        consommer, pas le recalculer.
    query_imports : imports actifs au niveau du curseur.
    chunk_symbols : symboles AST du chunk candidat (comparés à query_vars).
    chunk_imports : imports du chunk candidat (comparés à query_imports).
    doc_weights : poids IDF précalculés par symbole ; 1.0 si absent.

    Numérateur (intersection) : pour chaque symbole de la requête (variable
    ou import) aussi présent dans le chunk,
        poids = doc_weights.get(symbole, 1.0) * multiplicateur_de_type
    (le multiplicateur inclut le score d'attention pour les variables — les
    imports n'ont pas de notion de distance AST, donc pas d'attention_score,
    seulement leur propre multiplicateur `import_weight`).

    Dénominateur (union) : la somme ci-dessus pour TOUS les symboles de la
    requête (matchés ou non) + la somme des poids des symboles du chunk qui
    ne sont PAS dans la requête, où pour ceux-ci poids = doc_weights.get(v,
    1.0) tel quel (pas de multiplicateur de type, pas d'attention — lecture
    littérale de la spécification : seul le côté requête a un
    multiplicateur de type explicite).

    Toujours entre 0.0 et 1.0 (l'intersection est une somme partielle des
    termes déjà comptés côté requête dans l'union — jamais de terme compté
    en trop). Retourne 0.0 si requête et chunk sont tous les deux vides.
    """
    intersection_weight = 0.0
    union_weight = 0.0

    for var, attention_score in query_vars.items():
        weight = doc_weights.get(var, 1.0) * attention_score * var_weight
        union_weight += weight
        if var in chunk_symbols:
            intersection_weight += weight

    for imp in query_imports:
        weight = doc_weights.get(imp, 1.0) * import_weight
        union_weight += weight
        if imp in chunk_imports:
            intersection_weight += weight

    for symbol in chunk_symbols:
        if symbol not in query_vars:
            union_weight += doc_weights.get(symbol, 1.0)

    for imp in chunk_imports:
        if imp not in query_imports:
            union_weight += doc_weights.get(imp, 1.0)

    if union_weight == 0.0:
        return 0.0

    return intersection_weight / union_weight

In [ ]:
%%writefile retriever.py
import json
import re
from pathlib import Path
from typing import Any

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from ast_chunker import load_and_chunk_repo_ast_cached


def best_snippet(request, snippets):
    vectorizer = TfidfVectorizer()
    matrice_repo_snippets = vectorizer.fit_transform(snippets)
    matrice_request = vectorizer.transform([request])
    scores = cosine_similarity(matrice_request, matrice_repo_snippets)[0]
    best_index = scores.argmax()
    return snippets[best_index], scores[best_index]


def tokenize_code(code: str) -> set[str]:
    """Extraire le sac de mots (bag of words) d'un extrait de code."""
    return set(re.findall(r"\w+", code))


def jaccard_similarity(tokens_a: set[str], tokens_b: set[str]) -> float:
    """Calculer l'indice de Jaccard entre deux sacs de tokens."""
    union = tokens_a | tokens_b
    if not union:
        return 0.0
    return len(tokens_a & tokens_b) / len(union)


def retrieve_top_k_jaccard(
    incomplete_code: str,
    snippets: list[dict[str, str]],
    k: int = 10,
    exclude_task_id: str | None = None,
) -> list[dict[str, Any]]:
    """Classer les blocs de code par indice de Jaccard avec le code incomplet.

    Compare le sac de tokens du code incomplet à celui de chaque bloc
    (`{"file", "snippet"}`) et retourne les k meilleurs
    `{"file", "snippet", "score"}`, triés par score décroissant.

    `exclude_task_id`, si fourni, écarte les snippets extraits de cette
    tâche : leur `right_context` contient la suite réelle du code à
    compléter (voir `dataset.extract_repository_snippets`), donc les
    inclure dans la recherche pour cette même tâche fuiterait le
    groundtruth.
    """
    query_tokens = tokenize_code(incomplete_code)

    scored_snippets = [
        {
            "file": snippet.get("file"),
            "snippet": snippet["snippet"],
            "score": jaccard_similarity(query_tokens, tokenize_code(snippet["snippet"])),
        }
        for snippet in snippets
        if exclude_task_id is None or snippet.get("task_id") != exclude_task_id
    ]
    scored_snippets.sort(key=lambda item: item["score"], reverse=True)
    return scored_snippets[:k]


def load_repository_snippets(
    repository: str, repositories_dir: str | Path
) -> list[dict[str, str]]:
    """Charger les blocs de code sauvegardés (dataset.save_repository_snippets) d'un dépôt."""
    safe_name = re.sub(r"[^\w.-]", "_", repository)
    repository_path = Path(repositories_dir) / f"{safe_name}.jsonl"

    snippets = []
    with repository_path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                record = json.loads(line)
                snippets.append(
                    {
                        "file": record.get("file"),
                        "snippet": record["snippet"],
                        "task_id": record.get("task_id"),
                    }
                )
    return snippets


def retrieve_top_k_from_repository(
    incomplete_code: str,
    repository: str,
    repositories_dir: str | Path,
    k: int = 10,
    task_id: str | None = None,
) -> list[dict[str, Any]]:
    """Charger les blocs de code d'un dépôt puis retourner les k meilleurs (Jaccard).

    `task_id`, si fourni, exclut les snippets provenant de cette même tâche
    (voir `retrieve_top_k_jaccard`).
    """
    snippets = load_repository_snippets(repository, repositories_dir)
    return retrieve_top_k_jaccard(incomplete_code, snippets, k=k, exclude_task_id=task_id)


def retrieve_top_k_ast_jaccard(
    incomplete_code: str,
    chunks: list[dict[str, Any]],
    k: int = 10,
) -> list[dict[str, Any]]:
    """Classer des chunks AST (ast_chunker.chunk_file_ast/load_and_chunk_repo_ast)
    par indice de Jaccard entre le code incomplet et les identifiants de chaque
    fenêtre.

    Contrairement à `retrieve_top_k_jaccard`, qui compare le sac de tokens bruts
    du texte, la comparaison se fait ici contre `chunk["identifiers"]` : les
    noms de classes/fonctions/attributs/variables hérités des blocs AST
    (imports du module + tout bloc classe/fonction chevauchant la fenêtre) —
    docstrings et commentaires exclus.
    """
    query_tokens = tokenize_code(incomplete_code)

    scored_chunks = [
        {
            "file_path": chunk["file_path"],
            "line_start": chunk["line_start"],
            "line_end": chunk["line_end"],
            "raw_code": chunk["raw_code"],
            "score": jaccard_similarity(query_tokens, set(chunk["identifiers"])),
        }
        for chunk in chunks
    ]
    scored_chunks.sort(key=lambda item: item["score"], reverse=True)
    return scored_chunks[:k]


def retrieve_top_k_raw_jaccard(
    incomplete_code: str,
    chunks: list[dict[str, Any]],
    k: int = 10,
) -> list[dict[str, Any]]:
    """Classer les mêmes chunks AST par Jaccard sur le texte brut (`chunk["raw_code"]`),
    pas sur les identifiants.

    Existe pour comparer, à corpus et fenêtres strictement identiques (celles
    produites par `ast_chunker`), le signal de retrieval de l'officiel RG1
    (Jaccard sur tokens bruts, cf. `search_code.SimilarityScore.jaccard_similarity`)
    contre `retrieve_top_k_ast_jaccard` (Jaccard sur identifiants AST) — la seule
    variable qui change est le signal de scoring, tout le reste (chunking,
    anti-fuite, boucle) est partagé.
    """
    query_tokens = tokenize_code(incomplete_code)

    scored_chunks = [
        {
            "file_path": chunk["file_path"],
            "line_start": chunk["line_start"],
            "line_end": chunk["line_end"],
            "raw_code": chunk["raw_code"],
            "score": jaccard_similarity(query_tokens, tokenize_code(chunk["raw_code"])),
        }
        for chunk in chunks
    ]
    scored_chunks.sort(key=lambda item: item["score"], reverse=True)
    return scored_chunks[:k]


def retrieve_top_k_from_directory(
    incomplete_code: str,
    dir_path: str | Path,
    k: int = 10,
) -> list[dict[str, Any]]:
    """Découper tous les fichiers .py d'un dossier avec ast_chunker (mis en cache
    sur disque, voir `ast_chunker.load_and_chunk_repo_ast_cached`) puis retourner
    les k meilleurs chunks (Jaccard sur identifiants AST)."""
    chunks = load_and_chunk_repo_ast_cached(str(dir_path))
    return retrieve_top_k_ast_jaccard(incomplete_code, chunks, k=k)


def retrieve_top_k_for_dataset(
    records: list[dict[str, Any]],
    repositories_dir: str | Path,
    k: int = 10,
    query_lines: int | None = None,
) -> list[dict[str, Any]]:
    """Appliquer le retriever Jaccard à chaque exemple de line_completion.jsonl.

    La requête vient de `record["prompt"]` (le code incomplet), jamais du
    groundtruth ; si `query_lines` est fourni, seules les `query_lines`
    dernières lignes du prompt sont utilisées (cf. `dataset.last_lines`, S_s
    dans l'article RepoCoder). Les blocs de chaque dépôt sont tokenisés une
    seule fois (mis en cache) pour éviter de retokeniser à chaque exemple.
    """
    from dataset import last_lines

    results = []
    repository_index: dict[str, list[dict[str, Any]]] = {}

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository")
        if not repository:
            continue

        if repository not in repository_index:
            snippets = load_repository_snippets(repository, repositories_dir)
            repository_index[repository] = [
                {
                    "file": snippet["file"],
                    "snippet": snippet["snippet"],
                    "task_id": snippet.get("task_id"),
                    "tokens": tokenize_code(snippet["snippet"]),
                }
                for snippet in snippets
            ]

        task_id = metadata.get("task_id")
        query = record["prompt"] if query_lines is None else last_lines(record["prompt"], query_lines)
        query_tokens = tokenize_code(query)
        scored_snippets = [
            {
                "file": snippet["file"],
                "snippet": snippet["snippet"],
                "score": jaccard_similarity(query_tokens, snippet["tokens"]),
            }
            for snippet in repository_index[repository]
            if snippet.get("task_id") != task_id
        ]
        scored_snippets.sort(key=lambda item: item["score"], reverse=True)

        results.append(
            {
                "task_id": metadata.get("task_id"),
                "repository": repository,
                "retrieved_chunks": scored_snippets[:k],
            }
        )

    return results


if __name__ == "__main__":
    from dataset import SLIDING_STRIDE, load_jsonl, save_jsonl

    project_dir = Path(__file__).resolve().parent
    records = load_jsonl(r"C:\Users\User\Downloads\line_completion.jsonl")
    results = retrieve_top_k_for_dataset(
        records,
        project_dir / "data" / "repositories",
        k=10,
        query_lines=SLIDING_STRIDE,
    )
    save_jsonl(results, project_dir / "data" / "retrieved_jaccard.jsonl")

    print(f"{len(results)} exemples traités et sauvegardés dans data/retrieved_jaccard.jsonl")

In [ ]:
%%writefile single_turn_agent.py
import os
from typing import List, Set, Dict
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

# ==========================================
# 1. ANALYSTE AST (Tree-Sitter Extractor)
# ==========================================
class ASTContextAnalyzer:
    def __init__(self):
        self.py_language = Language(tspython.language())
        self.parser = Parser(self.py_language)

    def extract_context(self, code_snippet: str) -> Dict[str, Set[str]]:
        """
        Extrait en 0 ms les variables locales et les imports du code source.
        """
        if not code_snippet.strip():
            return {"variables": set(), "imports": set()}

        tree = self.parser.parse(bytes(code_snippet, "utf8"))
        variables = set()
        imports = set()

        stack = [tree.root_node]
        while stack:
            node = stack.pop()

            # Extraction des variables (Assignments)
            if node.type == "assignment":
                left_node = node.child_by_field_name("left")
                if left_node and left_node.type == "identifier":
                    var_name = code_snippet[left_node.start_byte:left_node.end_byte]
                    variables.add(var_name)

            # Extraction des Imports (import X / from X import Y)
            elif node.type in ("import_statement", "import_from_statement"):
                import_text = code_snippet[node.start_byte:node.end_byte].strip()
                imports.add(import_text)

            stack.extend(node.children)

        return {"variables": variables, "imports": imports}


# ==========================================
# 2. AGENT LLM MONO-TOUR (Single-Turn Agent)
# ==========================================
class SingleTurnCodeAgent:
    SYSTEM_PROMPT = (
        "Vous êtes un assistant de complétion de code en temps réel pour IDE.\n"
        "Analyse le code incomplet et utilise les exemples du dépôt pour prédire la suite.\n"
        "Règles strictes :\n"
        "1. Utilise en priorité les variables locales actives et les fonctions importées.\n"
        "2. N'invente pas de nouvelles signatures si des fonctions équivalentes existent dans les exemples.\n"
        "3. Génère UNIQUEMENT le code de complétion venant immédiatement après le curseur."
    )

    def __init__(self, llm_client, model_name: str = "qwen2.5-coder-7b"):
        """
        llm_client: Le client LLM (ex: OpenAI, Ollama, vLLM, HuggingFace)
        """
        self.client = llm_client
        self.model_name = model_name
        self.ast_analyzer = ASTContextAnalyzer()

    def build_structured_prompt(
        self,
        target_file_path: str,
        unfinished_code: str,
        retrieved_chunks: List[Dict[str, str]]
    ) -> str:
        """
        Construit le prompt enrichi avec les métadonnées de l'AST (Path, Imports, Variables)
        """
        # Analyse AST en temps réel sur le code incomplet
        ast_metadata = self.ast_analyzer.extract_context(unfinished_code)

        vars_str = ", ".join(ast_metadata["variables"]) if ast_metadata["variables"] else "None"
        imports_str = "\n".join(ast_metadata["imports"]) if ast_metadata["imports"] else "None"

        # Formate les exemples du dépôt récupérés par le Retriever Sparse
        retrieved_context_str = ""
        for i, chunk in enumerate(retrieved_chunks, 1):
            retrieved_context_str += f"\n# Example {i} | File: {chunk.get('file_path', 'unknown')}\n"
            retrieved_context_str += f"{chunk.get('raw_code', '')}\n"

        prompt = f"""=================== METADATA & CONTEXT ===================
FILE PATH: {target_file_path}

IMPORTS:
{imports_str}

ACTIVE SCOPE VARIABLES:
{vars_str}

=================== RETRIEVED REPO EXAMPLES =============
{retrieved_context_str}
=================== TARGET CODE TO COMPLETE =============
{unfinished_code}"""

        return prompt

    def generate_completion(
        self,
        target_file_path: str,
        unfinished_code: str,
        retrieved_chunks: List[Dict[str, str]]
    ) -> str:
        """
        Exécute la complétion en un seul tour (Single-Turn).
        """
        structured_prompt = self.build_structured_prompt(
            target_file_path, unfinished_code, retrieved_chunks
        )

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": structured_prompt}
            ],
            temperature=0.2,
            max_tokens=150
        )

        return response.choices[0].message.content

In [ ]:
%%writefile generator_min.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- copié tel quel depuis repocoder-mine/generator.py (call_huggingface_api),
#     + call_huggingface_instruct_api ajoutée pour les modèles chat/instruct ---

_hf_model_cache: dict = {}


def call_huggingface_api(
    prompt: str,
    model: str = "Salesforce/codegen-2B-mono",
    max_new_tokens: int = 64,
    temperature: float = 0.0,
    max_prompt_tokens: int = 4096,
) -> str:
    """Générer une complétion avec un modèle Hugging Face chargé localement (complétion brute)."""
    if model not in _hf_model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model)
        tokenizer.truncation_side = "left"
        hf_model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        hf_model.eval()
        _hf_model_cache[model] = (tokenizer, hf_model)

    tokenizer, hf_model = _hf_model_cache[model]
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=max_prompt_tokens
    ).to(hf_model.device)

    with torch.no_grad():
        output_ids = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


_hf_instruct_model_cache: dict = {}


def call_huggingface_instruct_api(
    prompt: str,
    system_prompt: str,
    model: str = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_new_tokens: int = 64,
    temperature: float = 0.0,
    max_prompt_tokens: int = 4096,
) -> str:
    """Comme call_huggingface_api, mais via tokenizer.apply_chat_template (modèle instruct)."""
    if model not in _hf_instruct_model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model)
        tokenizer.truncation_side = "left"
        hf_model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        hf_model.eval()
        _hf_instruct_model_cache[model] = (tokenizer, hf_model)

    tokenizer, hf_model = _hf_instruct_model_cache[model]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        chat_text, return_tensors="pt", truncation=True, max_length=max_prompt_tokens
    ).to(hf_model.device)

    with torch.no_grad():
        output_ids = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## 3. Télécharger le dataset officiel RepoCoder + les 8 vrais dépôts

In [ ]:
!git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
%cd codet_src
!git sparse-checkout init --cone
!git sparse-checkout set RepoCoder
!git checkout main
%cd ..

In [ ]:
import zipfile

with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
    z.extractall('datasets_rapo')
with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
    z.extractall('repos_source')

print('Dataset et dépôts extraits.')

## 4. Charger les tâches (échantillon reproductible)

In [ ]:
import json
import random
from pathlib import Path

REPOS_DIR = Path('repos_source')
TASKS_PATH = Path('datasets_rapo/line_level_completion_1k_context_codegen.test.jsonl')
N_TRIALS = 300
SEED = 42


def load_tasks(path):
    tasks = []
    with path.open('r', encoding='utf-8') as file:
        for line in file:
            if line.strip():
                tasks.append(json.loads(line))
    return tasks


all_tasks = load_tasks(TASKS_PATH)
sampled_tasks = random.Random(SEED).sample(all_tasks, N_TRIALS)
print(f'{len(all_tasks)} tâches disponibles, {len(sampled_tasks)} échantillonnées (seed={SEED}).')

## 5. Fonctions communes

Anti-fuite, troncature du code inachevé (`trim_code`, corrige le bug de contexte supprimé déjà diagnostiqué dans `colab_50_trials.ipynb`), construction du prompt, scoring EM/ES, nettoyage des balises markdown pour l'instruct.

In [ ]:
import os
import re
import editdistance


def filter_safe_chunks(chunks, repo_dir, fpath_tuple, context_start_lineno):
    task_file = os.path.normpath(str(repo_dir.joinpath(*fpath_tuple[1:])))
    safe = []
    for chunk in chunks:
        if os.path.normpath(chunk['file_path']) == task_file:
            if chunk['line_end'] - 1 > context_start_lineno:
                continue
        safe.append(chunk)
    return safe


def trim_code(text, max_chars=3000):
    if len(text) <= max_chars:
        return text
    trimmed = text[-max_chars:]
    newline_pos = trimmed.find('\n')
    return trimmed[newline_pos + 1:] if newline_pos != -1 else trimmed


def build_prompt(unfinished_code, retrieved_chunks, repo_dir, max_context_chars=3000):
    blocks = []
    total_chars = 0
    for chunk in retrieved_chunks:  # déjà trié du meilleur au moins bon
        rel_path = os.path.relpath(chunk['file_path'], repo_dir)
        block = f"# the below code fragment can be found in: {rel_path}\n{chunk['raw_code']}\n"
        if total_chars + len(block) > max_context_chars:
            break
        blocks.append(block)
        total_chars += len(block)
    context = '\n'.join(reversed(blocks))
    return f"{context}\n{unfinished_code}" if context else unfinished_code


def compute_em(target, prediction):
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    return int(target_lines == prediction_lines and len(target_lines) > 0)


def compute_es(target, prediction):
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    target_str = '\n'.join(target_lines)
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    prediction_str = '\n'.join(prediction_lines)
    if not target_str and not prediction_str:
        return 1.0
    return 1 - (editdistance.eval(target_str, prediction_str) / max(len(target_str), len(prediction_str), 1))


def strip_markdown_fence(text):
    text = text.strip()
    text = re.sub(r'^```[a-zA-Z0-9_+-]*\s*\n?', '', text)
    text = re.sub(r'\n?```\s*$', '', text)
    return text

## 6. Retriever pondéré : IDF par dépôt + séparation imports/symboles

Ce qui manquait pour appeler `weighted_ast_attention_score` sur de vraies données (voir `test_weighted_ast_retrieval.py` en local, même logique ici).

In [ ]:
import math
from collections import Counter

from ast_chunker import load_and_chunk_repo_ast_cached, build_scope_map
from ast_distance import compute_query_vars_with_attention
from weighted_ast_scorer import weighted_ast_attention_score
from retriever import retrieve_top_k_raw_jaccard, retrieve_top_k_ast_jaccard
from single_turn_agent import SingleTurnCodeAgent
from generator_min import call_huggingface_api, call_huggingface_instruct_api


def compute_doc_weights(chunks):
    n_docs = len(chunks)
    df = Counter()
    for chunk in chunks:
        df.update(set(chunk['identifiers']))
    return {symbol: math.log((n_docs + 1) / (count + 1)) + 1 for symbol, count in df.items()}


def compute_repo_import_names(repo_dir):
    import_names = set()
    for root, _, files in os.walk(repo_dir):
        for filename in files:
            if not filename.endswith('.py'):
                continue
            file_path = os.path.join(root, filename)
            try:
                code_text = open(file_path, 'r', encoding='utf-8').read()
                module_imports, _ = build_scope_map(code_text)
            except (SyntaxError, OSError):
                continue
            import_names |= module_imports
    return import_names


def split_chunk_symbols(chunk, repo_import_names):
    identifiers = set(chunk['identifiers'])
    chunk_imports = identifiers & repo_import_names
    chunk_symbols = identifiers - chunk_imports
    return chunk_symbols, chunk_imports


def retrieve_top_k_weighted(query_vars, query_imports, chunks, repo_import_names, doc_weights, k=10, var_weight=2.0, import_weight=2.5):
    scored = []
    for chunk in chunks:
        chunk_symbols, chunk_imports = split_chunk_symbols(chunk, repo_import_names)
        score = weighted_ast_attention_score(
            query_vars, query_imports, chunk_symbols, chunk_imports, doc_weights,
            var_weight=var_weight, import_weight=import_weight,
        )
        scored.append({**chunk, 'score': score})
    scored.sort(key=lambda item: item['score'], reverse=True)
    return scored[:k]

## 7. Boucle générique : 4 signaux de retrieval x 2 modèles

`retrieval_mode` : `'none'` / `'raw'` / `'ast'` / `'weighted'`. `model_family` : `'base'` (codegen, complétion brute) / `'instruct'` (Qwen, chat template + nettoyage des balises markdown).

In [ ]:
BASE_MODEL_NAME = 'Salesforce/codegen-2B-mono'
INSTRUCT_MODEL_NAME = 'Qwen/Qwen2.5-Coder-3B-Instruct'
BASE_MAX_PROMPT_TOKENS = 2048 - 64 - 32  # n_positions=2048 pour codegen-*-mono
INSTRUCT_MAX_PROMPT_TOKENS = 4096  # Qwen a une fenêtre bien plus large, pas la même contrainte

chunk_cache = {}
doc_weights_cache = {}
import_names_cache = {}


def run_condition(tasks, condition_name, retrieval_mode, model_family, model=None, k=10, max_code_chars=3000, lam=0.1, var_weight=2.0, import_weight=2.5, max_new_tokens=64):
    condition_results = []
    for i, task in enumerate(tasks, start=1):
        metadata = task['metadata']
        repo = metadata['task_id'].split('/')[0]
        repo_dir = REPOS_DIR / repo
        unfinished_code = trim_code(task['prompt'], max_chars=max_code_chars)

        if repo not in chunk_cache:
            chunk_cache[repo] = load_and_chunk_repo_ast_cached(str(repo_dir))

        if retrieval_mode == 'none':
            retrieved = []
        elif retrieval_mode == 'raw':
            safe_chunks = filter_safe_chunks(chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno'])
            retrieved = retrieve_top_k_raw_jaccard(unfinished_code, safe_chunks, k=k)
        elif retrieval_mode == 'ast':
            safe_chunks = filter_safe_chunks(chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno'])
            retrieved = retrieve_top_k_ast_jaccard(unfinished_code, safe_chunks, k=k)
        elif retrieval_mode in ('weighted', 'weighted_no_attention'):
            if repo not in doc_weights_cache:
                doc_weights_cache[repo] = compute_doc_weights(chunk_cache[repo])
                import_names_cache[repo] = compute_repo_import_names(repo_dir)
            target_file = repo_dir.joinpath(*metadata['fpath_tuple'][1:])
            file_source = target_file.read_text(encoding='utf-8')
            try:
                module_imports, blocks = build_scope_map(file_source)
                line_no = metadata['line_no']
                candidate_vars = set()
                for block in blocks:
                    if block.line_start <= line_no <= block.line_end:
                        candidate_vars |= block.identifiers
                candidate_vars -= module_imports
                query_vars = compute_query_vars_with_attention(file_source, line_no, candidate_vars, lam=lam)
            except SyntaxError:
                # le fichier CIBLE de la tache lui-meme n'est pas du Python valide (ex.
                # template cookiecutter avec des balises Jinja {{ ... }}) -> impossible
                # d'extraire variables/imports par AST pour cette tache precise. Repli sur
                # une requete vide plutot que de planter tout le run (les modes raw/ast
                # n'ont pas ce probleme car ils n'ont pas besoin de parser le fichier cible).
                print(f"[{condition_name}] fichier cible non parsable, requete vide: {target_file}")
                module_imports, query_vars = set(), {}
            if retrieval_mode == 'weighted_no_attention':
                # mêmes variables (celles réellement trouvées avant le curseur), mais poids
                # d'attention forcé à 1.0 -> isole l'effet de l'attention de distance AST,
                # à IDF et pondération par type identiques
                query_vars = {var: 1.0 for var in query_vars}
            safe_chunks = filter_safe_chunks(chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno'])
            retrieved = retrieve_top_k_weighted(
                query_vars, module_imports, safe_chunks, import_names_cache[repo], doc_weights_cache[repo], k=k,
                var_weight=var_weight, import_weight=import_weight,
            )
        else:
            raise ValueError(f'retrieval_mode inconnu: {retrieval_mode}')

        prompt = build_prompt(unfinished_code, retrieved, repo_dir) if retrieved else unfinished_code

        if model_family == 'base':
            base_max_prompt_tokens = 2048 - max_new_tokens - 32  # n_positions=2048 pour codegen-*-mono
            completion_raw = call_huggingface_api(
                prompt, model=model or BASE_MODEL_NAME, max_new_tokens=max_new_tokens, max_prompt_tokens=base_max_prompt_tokens
            )
            completion = completion_raw
        elif model_family == 'instruct':
            completion_raw = call_huggingface_instruct_api(
                prompt, SingleTurnCodeAgent.SYSTEM_PROMPT, model=model or INSTRUCT_MODEL_NAME,
                max_new_tokens=max_new_tokens, max_prompt_tokens=INSTRUCT_MAX_PROMPT_TOKENS,
            )
            completion = strip_markdown_fence(completion_raw)
        else:
            raise ValueError(f'model_family inconnu: {model_family}')

        em = compute_em(metadata['ground_truth'], completion)
        es = compute_es(metadata['ground_truth'], completion)
        condition_results.append({
            'task_id': metadata['task_id'],
            'ground_truth': metadata['ground_truth'],
            'completion': completion,
            'completion_raw': completion_raw,
            'exact_match': em,
            'edit_similarity': es,
        })
        if i % 10 == 0 or i == len(tasks):
            print(f'[{condition_name}] {i}/{len(tasks)} — EM cumulé: '
                  f"{sum(r['exact_match'] for r in condition_results)}/{i}")
    return condition_results


def summarize(name, condition_results):
    em = sum(r['exact_match'] for r in condition_results)
    es = sum(r['edit_similarity'] for r in condition_results) / len(condition_results)
    n = len(condition_results)
    se = (em / n * (1 - em / n) / n) ** 0.5
    print(f"{name:<32}{em}/{n} ({100 * em / n:.1f}% ± {100 * 1.96 * se:.1f} pts, IC95%)"
          f"{'':>3}{es:>10.3f}")

## 8. Lancer les 8 conditions (4 signaux x 2 modèles)

Charge d'abord `codegen-2B-mono`, fait les 4 conditions base, PUIS charge `Qwen2.5-Coder-3B-Instruct` et fait les 4 conditions instruct (les deux modèles restent en cache simultanément si la mémoire GPU le permet — sinon redémarre la session entre les deux blocs).

In [ ]:
results = {}

for retrieval_mode in ['none', 'raw', 'ast', 'weighted_no_attention', 'weighted']:
    key = f'base_{retrieval_mode}'
    results[key] = run_condition(sampled_tasks, key, retrieval_mode, model_family='base')


In [ ]:
# Instruct désactivé pour ce run (5 conditions x 300 tâches sur le seul modèle de
# base représente déjà 1500 générations) — décommente si tu veux comparer aussi.
# for retrieval_mode in ['none', 'raw', 'ast', 'weighted_no_attention', 'weighted']:
#     key = f'instruct_{retrieval_mode}'
#     results[key] = run_condition(sampled_tasks, key, retrieval_mode, model_family='instruct')


## 9. Résultats

In [ ]:
for key, res in results.items():
    with open(f'results_{key}.jsonl', 'w', encoding='utf-8') as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"{'':<36}{'Exact Match (IC95%)':>32}{'Edit Similarity':>18}")
labels = {
    'none': 'sans retrieval',
    'raw': 'Jaccard texte brut',
    'ast': 'Jaccard identifiants AST (sans attention, sans IDF)',
    'weighted_no_attention': 'Jaccard pondéré (IDF + type), sans attention',
    'weighted': 'Jaccard pondéré + attention AST',
}
for model_family in ['base', 'instruct']:
    keys_present = [f'{model_family}_{m}' for m in labels if f'{model_family}_{m}' in results]
    if not keys_present:
        continue
    print(f'--- {model_family} ---')
    for retrieval_mode in ['none', 'raw', 'ast', 'weighted_no_attention', 'weighted']:
        key = f'{model_family}_{retrieval_mode}'
        if key in results:
            summarize(labels[retrieval_mode], results[key])


## 10. Balayage d'hyperparamètres (modèle de base)

Le retriever pondéré est en tête sur le modèle de base (39.0% à n=200) mais pas encore significativement détaché des deux retrievers plus simples. On teste quelques combinaisons de `lambda` (vitesse de décroissance de l'attention avec la distance AST) et des poids par type (`var_weight`, `import_weight`), sur un **sous-échantillon de 50 tâches** (les 50 premières de `sampled_tasks`, réutilisées) pour rester rapide — une génération complète à 200 tâches par config serait trop coûteuse en temps GPU pour un simple balayage exploratoire. Une fois une config prometteuse repérée, on la validera à 200 tâches séparément.

Intuition sur `lambda` : plus il est petit, plus l'attention décroît lentement avec la distance — les variables lointaines dans l'arbre gardent un poids significatif. Plus il est grand, seules les variables très proches du curseur comptent vraiment.

In [ ]:
sweep_sample = sampled_tasks[:50]

sweep_configs = [
    {'name': 'défaut (lam=0.1, var=2.0, imp=2.5)', 'lam': 0.1, 'var_weight': 2.0, 'import_weight': 2.5},
    {'name': 'lambda plus petit (lam=0.05)',        'lam': 0.05, 'var_weight': 2.0, 'import_weight': 2.5},
    {'name': 'lambda plus grand (lam=0.3)',         'lam': 0.3, 'var_weight': 2.0, 'import_weight': 2.5},
    {'name': 'poids égaux (var=imp=1.0)',           'lam': 0.1, 'var_weight': 1.0, 'import_weight': 1.0},
    {'name': 'imports dominants (var=1.0, imp=4.0)', 'lam': 0.1, 'var_weight': 1.0, 'import_weight': 4.0},
    {'name': 'variables dominantes (var=4.0, imp=1.0)', 'lam': 0.1, 'var_weight': 4.0, 'import_weight': 1.0},
]

sweep_results = {}
for cfg in sweep_configs:
    key = cfg['name']
    sweep_results[key] = run_condition(
        sweep_sample, f'sweep: {key}', 'weighted', model_family='base',
        lam=cfg['lam'], var_weight=cfg['var_weight'], import_weight=cfg['import_weight'],
    )

## 11. Résultat du balayage

In [ ]:
print(f"{'Config':<38}{'Exact Match':>15}{'Edit Similarity':>18}")
for cfg in sweep_configs:
    key = cfg['name']
    res = sweep_results[key]
    em = sum(r['exact_match'] for r in res)
    es = sum(r['edit_similarity'] for r in res) / len(res)
    print(f"{key:<38}{em}/{len(res)} ({100*em/len(res):.1f}%){'':>3}{es:>15.3f}")

print()
print('Pour rappel (référence à ce même sous-échantillon de 50 tâches) :')
print('les résultats base_none/base_raw/base_ast/base_weighted (config défaut) de la section 9')

## 12. Valider la meilleure config à 200 tâches

Une fois une config gagnante identifiée en section 11, ajuste les valeurs ci-dessous et relance sur les 200 tâches complètes pour confirmer que le gain tient à plus grande échelle.

In [ ]:
BEST_LAM = 0.1          # à ajuster selon le résultat du balayage
BEST_VAR_WEIGHT = 2.0
BEST_IMPORT_WEIGHT = 2.5

results['base_weighted_tuned'] = run_condition(
    sampled_tasks, 'base_weighted_tuned', 'weighted', model_family='base',
    lam=BEST_LAM, var_weight=BEST_VAR_WEIGHT, import_weight=BEST_IMPORT_WEIGHT,
)
summarize('pondéré (config ajustée)', results['base_weighted_tuned'])
summarize('pondéré (config défaut, référence)', results['base_weighted'])

## 13. Dataset alternatif : `api_level_completion` (complétion d'appel API)

Même structure que `line_level_completion` (mêmes clés, mêmes 8 dépôts), mais le `ground_truth` est souvent un appel de fonction/méthode complet plutôt qu'une simple ligne — **24.6% des tâches ont un ground_truth multi-lignes** (moyenne 2.1 lignes, max 22). `max_new_tokens=64` (calibré pour du 1-ligne) risquerait de couper des complétions valides en plein milieu — on monte à **128**, avec le budget de prompt réduit en conséquence pour rester sous la fenêtre de 2048 tokens de `codegen`.

Échantillon séparé de `sampled_tasks` (dataset différent), taille plus modeste pour commencer (ce dataset n'a pas encore été testé de bout en bout).

In [ ]:
API_TASKS_PATH = Path('datasets_rapo/api_level_completion_1k_context_codegen.test.jsonl')
N_TRIALS_API = 300

api_all_tasks = load_tasks(API_TASKS_PATH)
api_sampled_tasks = random.Random(SEED).sample(api_all_tasks, N_TRIALS_API)
print(f'{len(api_all_tasks)} tâches api_level disponibles, {len(api_sampled_tasks)} échantillonnées.')

## 14. Lancer les 5 conditions sur `api_level_completion` (modèle de base)

In [ ]:
api_results = {}
for retrieval_mode in ['none', 'raw', 'ast', 'weighted_no_attention', 'weighted']:
    key = f'api_base_{retrieval_mode}'
    api_results[key] = run_condition(
        api_sampled_tasks, key, retrieval_mode, model_family='base', max_new_tokens=128
    )

## 15. Résultats — api_level_completion

In [ ]:
for key, res in api_results.items():
    with open(f'results_{key}.jsonl', 'w', encoding='utf-8') as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"{'':<52}{'Exact Match (IC95%)':>32}{'Edit Similarity':>18}")
api_labels = {
    'none': 'sans retrieval',
    'raw': 'Jaccard texte brut',
    'ast': 'Jaccard identifiants AST (sans attention, sans IDF)',
    'weighted_no_attention': 'Jaccard pondéré (IDF + type), sans attention',
    'weighted': 'Jaccard pondéré + attention AST',
}
for retrieval_mode in ['none', 'raw', 'ast', 'weighted_no_attention', 'weighted']:
    summarize(api_labels[retrieval_mode], api_results[f'api_base_{retrieval_mode}'])